In [1]:
from PIL import Image
import PIL
import numpy as np
import pandas as pd
from ScratchTorch.grad import Tensor
import os
from IPython.display import display

In [2]:
# convulution layer shown mathematically
kernel_size = 2
in_channels = 1
out_channels = 1
# input * conv2d layer 
np.random.uniform(-1,1,(in_channels,28,28))[:,:kernel_size,:kernel_size] * np.random.uniform(-1,1,(out_channels,in_channels,kernel_size,kernel_size))

array([[[[-0.0993045 ,  0.50815121],
         [-0.14821281,  0.31105278]]]])

# Data Loading

In [3]:
dataset_path = r'dataset/'

In [4]:
# loading the images from csv and converting it to image array. 
def convert_csv_to_img(path_to_csv,output_folder):
    data_csv = pd.read_csv(path_to_csv)
    labels = data_csv['label'].to_numpy()
    pixels = data_csv.drop(columns="label").values
    
    # Create the root output directory
    os.makedirs(output_folder, exist_ok=True)
    
    for idx, (label,pixel_row) in  enumerate(zip(labels,pixels)):
        img_array = pixel_row.reshape(28,28).astype(np.uint8)
        
        # creating folder for each class image
        class_dir = os.path.join(output_folder,str(label))
        os.makedirs(class_dir,exist_ok=True) #creating the folder if doesnt exists
        
        # converting the array to image and saving it
        img = Image.fromarray(img_array, mode='L')
        img.save(os.path.join(class_dir, f"digit_{idx}.png"))
        
        # displaying progress at every 10% interval.
        if (idx + 1) % (len(pixels)*.1) == 0:
            print(f"Processed {idx + 1} images...")

    print(f"Done! Images saved to '{output_folder}' organized by digit subfolders.")
    

In [5]:
# convert_csv_to_img('dataset/train.csv','dataset/train_images/') #only need to be executed once.

In [6]:
class ImageDataset:
    def __init__(self,files):
        self.files = files
        print(f"{len(self.files)} images loaded. ")
        
        
    def __len__(self):
        return len(self.files)
    
    def __getitem__(self, idx):
        img = Image.open(self.files[idx][0])
        img = np.array(img,dtype=np.float32) 
        img_tensor = Tensor(img)
        label = self.files[idx][1]
        return img_tensor,label
    
    
    @staticmethod
    def load_files(root_dir,has_classes=True, type_of_img = ('.png','.jpg','.jpeg'),train_test_split_ratio=.8):
        files = []
        if has_classes:
            for classes in os.listdir(root_dir):
                for f in os.listdir(os.path.join(root_dir, classes)):
                    if f.endswith((type_of_img)):
                        files.append((os.path.join(root_dir,classes,f),classes))
                        
            np.random.shuffle(files)
            # returning train and test files
            return files[:int(train_test_split_ratio*len(files))],files[int(train_test_split_ratio*len(files)):]
    
    
        

In [7]:
train_files,test_files = ImageDataset.load_files('dataset/train_images')

train_dataset = ImageDataset(train_files)
test_dataset = ImageDataset(test_files)


33600 images loaded. 
8400 images loaded. 


In [8]:
img_tensor, label = train_dataset[12]
print(label)
display(Image.fromarray(img_tensor.normalize_image_array())) #28x28 image

2


# Dataset Loader in batches.

In [9]:
class DatasetLoader:
    def __init__(self,dataset,batch_size=64, shuffle=True):
        self.dataset = dataset
        self.batch_size = batch_size
        self.shuffle = shuffle
        
    def __iter__(self):
        indices = list(range(len(self.dataset)))
        if self.shuffle:
            np.random.shuffle(indices)
            
        for start in range(0,len(indices),self.batch_size):
            batch_idx = indices[start:start+self.batch_size]
            images = []
            labels = []
            for idx in batch_idx:
                img, label = self.dataset[idx]
                images.append(img)
                labels.append(label)

            
            yield np.stack(images),np.array(labels)

In [10]:
train_loader = DatasetLoader(train_dataset)

# testing the loader
for batch,batch_label in train_loader:
    batch_tensor = Tensor.stack(batch)
    print(type(batch_tensor))
    break

<class 'ScratchTorch.grad.Tensor'>


In [11]:
import ScratchTorch.nn as nn
class CNNgoesBrr:
    def __init__(self,layers):
        if self.validate_layers(layers):
            self.layers= layers
        else:
            raise ValueError("Model Could not be initialised.")
        
    def __call__(self, x):
        for layer in self.layers:
                x = layer(x)
        return x
    
    
    def validate_layers(self,layers):
        for idx,layer in enumerate(layers):
            if not isinstance(layer,(nn.Conv2d, nn.Dense,nn.Flatten)):
                raise ValueError(f"Invalid Layer Type passed to CNN Model. dtype:{type(layer)}. Layer Number:{idx+1}")
        return True
        
        

In [12]:
cnn_model = CNNgoesBrr([nn.Conv2d(1,16,3),
                        nn.Conv2d(16,8,3),
                        nn.Flatten(),
                        nn.Dense(4608,8).set_activation_function('tanh'),
                        nn.Dense(8,8).set_activation_function('tanh'),
                        nn.Dense(8,10).set_activation_function('softmax')])

In [13]:
cnn_layer = nn.Conv2d(1,16,5)

In [15]:
cnn_model(img_tensor.reshape((1,1,28,28)))

Tensor(data=[[0.11884334 0.14556359 0.06133274 0.12864119 0.06025219 0.11789819
  0.07122523 0.06395854 0.14313825 0.08914674]])

In [ ]:
for batch,batch_label in train_loader:
    batch_tensor = Tensor.stack(batch)
    batch_tensor = batch_tensor.reshape((64,1,28,28))
    outputs = np.eye(10)[batch_label.astype(int)]
    out = cnn_model(batch_tensor)
    print(out[0])
    print(f"shape of Y after passing one batch:{out.shape}")
    break

In [ ]:
print(out)

Tensor(data=[[0.18059439 0.11406577 0.07708914 0.13415062 0.07920082 0.04933545
  0.08041563 0.19526929 0.04607885 0.04380003]])


In [ ]:
Tensor.stack(batch).shape

(64, 28, 28)